# DarkTech Remote Worker (Colab GPU)

Roda o worker de geração de áudio na GPU do Colab e expõe via Cloudflare Tunnel.

Sua UI Gradio local (no VSCode) chama essa URL quando precisa renderizar um stem.
Mastering, orquestração com DeepSeek e a UI ficam locais.

**Antes de rodar**: Runtime > Change runtime type > GPU (T4 grátis basta; A100 com Pro).

In [ ]:
# 1/4: API key para esta sessão
import os, secrets

os.environ['DARKTECH_WORKER_API_KEY'] = secrets.token_urlsafe(24)
print('API key desta sessão:')
print('   ', os.environ['DARKTECH_WORKER_API_KEY'])
print()
print('Copie para o seu .env local como DARKTECH_REMOTE_API_KEY')

os.environ.setdefault('HF_TOKEN', '')  # opcional, sobe limites de download HF

In [ ]:
# 2/4: clonar o repo, instalar deps, montar Drive, instalar cloudflared
import os, subprocess, shutil, sys
from pathlib import Path

REPO_URL = 'https://github.com/horningwalter/gorit-lab-darktech-generator.git'
REPO_DIR = Path('/content/gorit_lab_darktech_generator')

if not REPO_DIR.exists():
    subprocess.check_call(['git', 'clone', '-q', REPO_URL, str(REPO_DIR)])
else:
    subprocess.check_call(['git', '-C', str(REPO_DIR), 'pull', '-q'])
%cd /content/gorit_lab_darktech_generator

print('Instalando pacote (~1-2 min)...')
res = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[ml]',
     'fastapi', 'uvicorn[standard]'],
    capture_output=True, text=True,
)
if res.stdout:
    print(res.stdout[-2000:])
if res.returncode != 0:
    print('PIP FALHOU:')
    print(res.stderr[-3000:])
    raise SystemExit(1)
print('Pip OK.')

try:
    import darktech_generator
    print('darktech_generator importado, versão', darktech_generator.__version__)
except Exception as e:
    print('IMPORT FALHOU:', e)
    raise

from darktech_generator.colab.bootstrap import bootstrap
import json
print(json.dumps(bootstrap(), indent=2))

if not shutil.which('cloudflared'):
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb -O /tmp/cf.deb
    !sudo dpkg -i /tmp/cf.deb 2>&1 | tail -2
print('cloudflared:', shutil.which('cloudflared'))

In [ ]:
# 3/4: subir o worker FastAPI em background e checar /health
import subprocess, sys, time, urllib.request, json, os
from pathlib import Path

worker_log = Path('/tmp/worker.log')
worker = subprocess.Popen(
    [sys.executable, '-m', 'uvicorn',
     'darktech_generator.generation.remote_worker:build_app',
     '--factory', '--host', '127.0.0.1', '--port', '8000'],
    stdout=worker_log.open('w'), stderr=subprocess.STDOUT,
)
print('Worker PID', worker.pid, '(logs em /tmp/worker.log)')

ok = False
for i in range(40):
    if worker.poll() is not None:
        print('WORKER MORREU. Últimas linhas do log:')
        print(worker_log.read_text()[-2000:])
        raise SystemExit(1)
    try:
        req = urllib.request.Request(
            'http://127.0.0.1:8000/health',
            headers={'X-API-Key': os.environ['DARKTECH_WORKER_API_KEY']},
        )
        with urllib.request.urlopen(req, timeout=2) as r:
            print('/health:', json.dumps(json.loads(r.read()), indent=2))
            ok = True
            break
    except Exception:
        time.sleep(1)
if not ok:
    print('Worker não respondeu em 40s. Últimas linhas do log:')
    print(worker_log.read_text()[-2000:])
    raise SystemExit(1)
print('Worker pronto em http://127.0.0.1:8000')

In [ ]:
# 4/4: subir Cloudflare Tunnel e capturar URL pública
# Deixe esta célula rodando enquanto usa a UI local. Para encerrar, pare a célula.
import subprocess, re, time
from pathlib import Path

tunnel_log = Path('/tmp/cloudflared.log')
if tunnel_log.exists():
    tunnel_log.unlink()

tunnel = subprocess.Popen(
    ['cloudflared', 'tunnel', '--no-autoupdate',
     '--url', 'http://127.0.0.1:8000',
     '--logfile', str(tunnel_log),
     '--loglevel', 'info'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
)

url_re = re.compile(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com')
public_url = None

print('Aguardando URL do Cloudflare (até 60s)...', flush=True)
deadline = time.time() + 60
while public_url is None and time.time() < deadline:
    if tunnel.poll() is not None:
        print('cloudflared morreu. Log:')
        print(tunnel_log.read_text()[-2000:] if tunnel_log.exists() else '(sem log)')
        raise SystemExit(1)
    line = tunnel.stdout.readline()
    if line:
        print(line.rstrip(), flush=True)
        m = url_re.search(line)
        if m:
            public_url = m.group(0)
            break
    else:
        if tunnel_log.exists():
            m = url_re.search(tunnel_log.read_text())
            if m:
                public_url = m.group(0)
                break
        time.sleep(0.3)

if public_url is None:
    print('Não consegui extrair a URL em 60s. Conteúdo do log:')
    print(tunnel_log.read_text() if tunnel_log.exists() else '(sem log)')
    raise SystemExit(1)

print('\n=====================================================')
print(' Cloudflare Tunnel pronto')
print(' Cole no .env local da sua máquina:')
print(f'   DARKTECH_REMOTE_URL={public_url}')
print('=====================================================\n')
print('Mantendo célula viva. Pare a execução para encerrar o tunnel.', flush=True)

try:
    for line in iter(tunnel.stdout.readline, ''):
        print(line.rstrip(), flush=True)
except KeyboardInterrupt:
    tunnel.terminate()
    print('Tunnel encerrado.')